# **Cài đặt thư viện**

In [1]:
!pip install requests beautifulsoup4 pandas tqdm

# **Import thư viện và tạo file**

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
import os

os.makedirs("data", exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}

# **CRAWL REAL (VnExpress)**

## **Lấy link bài báo**

In [3]:
def get_links(page_url):
    try:
        res = requests.get(page_url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "vnexpress.net" in href and ".html" in href:
                links.append(href)

        return list(set(links))  # bỏ trùng
    except:
        return []

## **Lấy nội dung bài viết**

In [4]:
def get_article(url):
    try:
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        title = soup.find("h1")
        if not title:
            return None

        paragraphs = soup.find_all("p")
        text = " ".join([p.text for p in paragraphs])

        # lọc bài quá ngắn
        if len(text) < 200:
            return None

        return text
    except:
        return None

## **Crawl REAL (có phân trang)**

In [ ]:
data_real = []

categories = [
    "thoi-su",
    "the-gioi",
    "giai-tri",
    "giao-duc",
    "kinh-doanh",
    "the-thao",
    "khoa-hoc-cong-nghe",
    "suc-khoe"
]

MAX_PAGES = 30   # tăng lên 50 nếu muốn nhiều hơn

for cat in categories:
    print(f"===== CATEGORY: {cat} =====")

    for i in range(1, MAX_PAGES + 1):
        page_url = f"https://vnexpress.net/{cat}-p{i}"
        print("Page:", page_url)

        links = get_links(page_url)

        for link in tqdm(links):
            text = get_article(link)

            if text:
                data_real.append({
                    "text": text,
                    "label": "REAL",
                    "source": "vnexpress"
                })

            time.sleep(0.5)  # tránh block

# chuyển thành dataframe
df_real = pd.DataFrame(data_real)

# bỏ trùng
df_real = df_real.drop_duplicates(subset=["text"])

# lưu file
df_real.to_csv("data/real_big.csv", index=False)

print("Số bài REAL:", len(df_real))

===== CATEGORY: thoi-su =====
Page: https://vnexpress.net/thoi-su-p1


100%|██████████| 94/94 [02:43<00:00,  1.74s/it]


Page: https://vnexpress.net/thoi-su-p2


100%|██████████| 60/60 [01:54<00:00,  1.91s/it]


Page: https://vnexpress.net/thoi-su-p3


...

100%|██████████| 60/60 [02:13<00:00,  2.23s/it]


Page: https://vnexpress.net/thoi-su-p13


100%|██████████| 60/60 [02:18<00:00,  2.30s/it]


Page: https://vnexpress.net/thoi-su-p14


 40%|████      | 24/60 [00:56<01:25,  2.38s/it]

# **CRAWL FAKE (tingia)**

## **Hàm lấy link bài báo tin giả**

In [ ]:
def get_links_tingia(page_url):
    try:
        res = requests.get(page_url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "tingia.gov.vn" in href:
                links.append(href)

        return list(set(links))
    except:
        return []

## **Lấy nội dung**

In [ ]:
def get_article_tingia(url):
    try:
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        title = soup.find("h1")
        if not title:
            return None

        paragraphs = soup.find_all("p")
        text = " ".join([p.text for p in paragraphs])

        if len(text) < 100:
            return None

        return text
    except:
        return None

## **Crawl FAKE**

In [ ]:
data_fake = []

pages = [
    "https://tingia.gov.vn/linh-vuc"
]

for page in pages:
    print("Crawl:", page)

    links = get_links_tingia(page)

    for link in tqdm(links):
        text = get_article_tingia(link)

        if text:
            data_fake.append({
                "text": text,
                "label": "FAKE",
                "source": "tingia"
            })

        time.sleep(1)

df_fake = pd.DataFrame(data_fake)
df_fake = df_fake.drop_duplicates(subset=["text"])

df_fake.to_csv("data/fake.csv", index=False)

print("Số bài FAKE gốc:", len(df_fake))

## **Tăng data Fake**

**Tạo fake synthetic**

In [ ]:
def make_fake(text):
    return "SỐC!!! " + text + " !!! KHẨN!!! CHIA SẺ NGAY"

**Augment dữ liệu**

In [ ]:
augmented = []

for t in df_fake["text"]:
    augmented.append(make_fake(t))
    augmented.append("TIN NÓNG: " + t)
    augmented.append("CẢNH BÁO!!! " + t)

df_aug = pd.DataFrame({
    "text": augmented,
    "label": "FAKE",
    "source": "synthetic"
})

df_fake_big = pd.concat([df_fake, df_aug])
df_fake_big = df_fake_big.drop_duplicates(subset=["text"])

df_fake_big.to_csv("data/fake_big.csv", index=False)

print("FAKE sau augment:", len(df_fake_big))

# **GỘP DATASET**

In [ ]:
df = pd.concat([df_real, df_fake_big])

df = df.drop_duplicates(subset=["text"])
df = df.sample(frac=1).reset_index(drop=True)

df.to_csv("data/final.csv", index=False)

print("TOTAL DATA:", len(df))

In [ ]:
from google.colab import files

files.download("data/final.csv")